# Revision de los datos de restauracion

Fase de mirar y decidir. **Este cuaderno no escribe ningun CSV**: cuando todo este cerrado, se
pasa a un script y se genera el `gold`.

## La fuente: solo el censo municipal

De las tres que habia sobre la mesa, se descartan dos:

| Fuente | Decision | Motivo |
|---|---|---|
| **Censo comercial del Ajuntament** | **se usa** | foto con fecha, 31/12/2024 |
| OpenStreetMap | descartada en la ciudad | cuenta 7.430 locales frente a 10.100 del censo, un 26% menos, y el fichero no trae fecha de cada elemento |
| Terrazas | descartada | solo mide las sillas de la calle, no el aforo total: un bar con 8 sillas fuera puede tener 60 plazas dentro |

El censo es de **2024 y no hay nada mas nuevo**: la serie publicada es 2014, 2016, 2019, 2022 y
2024, confirmado en la web del Ajuntament, en Open Data BCN y en el portal estadistico. Para 2026
esta anunciado el IDUL, un identificador unico de local; si sale, cruzar el censo con cualquier otro
registro municipal dejara de ser un problema.


In [1]:
# 1. Carga
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
RUTA = (RAIZ / "data" / "raw" / "restauracion_hoteles_provincia" /
        "bcn_cens_comercial_restauracion_2024.csv")

censo = pd.read_csv(RUTA, low_memory=False)
print(f"{len(censo):,} locales x {censo.shape[1]} columnas")
print(f"grupo de actividad: {censo['Nom_Grup_Activitat'].unique()}")

10,864 locales x 49 columnas
grupo de actividad: <StringArray>
['Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)']
Length: 1, dtype: str


## 1. Que categorias trae, y cuales son restauracion

El fichero viene filtrado por grupo de actividad, y ese grupo mete en el mismo saco la restauracion
y el alojamiento. Dentro hay mas variedad de la que parece.

In [2]:
# 2. Todas las categorias, sin filtrar nada todavia
reparto = censo["Nom_Activitat"].value_counts().rename("locales").to_frame()
reparto["%"] = (reparto["locales"] / len(censo) * 100).round(1)
display(reparto)

,locales,%
Nom_Activitat,,
Restaurants,4430,40.8
Bars / CIBERCAFÈ,4273,39.3
Serveis de menjar take away MENJAR RÀPID,778,7.2
serveis d'allotjament,764,7.0
Bars especials amb actuació / Bars musicals / Discoteques /PUB,387,3.6
Xocolateries / Geladeries / Degustació,148,1.4
serveis de menjar i begudes,74,0.7
altres,6,0.1
Altres ( per exemple VENDING),4,0.0


In [3]:
# 3. Que hay dentro de las categorias dudosas
#
# Los recuentos no bastan para decidir: hay que ver que locales son. Tres de estas categorias no
# son restauracion aunque esten en el mismo grupo de actividad.
for actividad in ["serveis de menjar i begudes", "altres", "Altres ( per exemple VENDING)"]:
    g = censo[censo["Nom_Activitat"] == actividad]
    print(f"-- {actividad}  ({len(g)} locales)")
    for nombre in g["Nom_Local"].head(6):
        print(f"     {nombre}")
    print()

-- serveis de menjar i begudes  (74 locales)
     BANCO DE BOQUERONES
     SN
     SN
     SN
     SN
     SN

-- altres  (6 locales)
     CHIQUITA ROOM
     SN
     LOTERIAS Y APUESTAS ADMIN NUM 287
     LOTERIAS Y APUESTAS DEL ESTADO 236
     NONETES
     LOTERIAS Y APUESTAS DEL ESTADO

-- Altres ( per exemple VENDING)  (4 locales)
     ABIERTO 25 HORAS
     SN
     BCN 24 HORES 
     ABIERTO 25 HORAS



## 2. La seleccion

`INCLUIDOS` es la unica linea que hay que tocar para cambiar el alcance. Se declara aqui arriba y
no repartida por el cuaderno, para que no haya dos sitios donde mirar.

In [4]:
# 4. Que entra y que no
#
# Entran bares y restaurantes, que es el objeto del analisis.
#
# Quedan FUERA, cada uno por su motivo:
#   allotjament          -> son hoteles, cuentan en el otro lado del analisis
#   altres               -> administraciones de loteria, no es restauracion
#   Altres (VENDING)     -> tiendas de 24 horas y maquinas
#   menjar i begudes     -> cajon de sastre: 73 de sus 74 locales no tienen ni nombre
#
# Y quedan fuera POR DECISION, no por ser otra cosa. Se listan aparte porque son restauracion de
# pleno derecho y volver a meterlas es cambiar una linea:
#   take away / menjar rapid            778 locales
#   bars musicals / discoteques / pub   387 locales
#   xocolateries / geladeries           148 locales

# Las categorias NO se escriben literales. El censo las escribe con espacios dobles y acentos
# --`Bars   / CIBERCAFÉ`-- y una comparacion literal falla en silencio: la primera version de esta
# celda dejaba fuera los 4.273 bares y el cuaderno terminaba sin quejarse. Se buscan por palabra
# clave y se comprueba que cada una encuentre exactamente una categoria.
import unicodedata


def buscar(clave: str) -> str:
    """Devuelve la categoria del censo que contiene `clave`. Falla si no hay exactamente una."""
    def limpiar(s):
        s = unicodedata.normalize("NFKD", str(s).lower())
        return "".join(c for c in s if not unicodedata.combining(c))

    encontradas = [a for a in censo["Nom_Activitat"].unique() if clave in limpiar(a)]
    if len(encontradas) != 1:
        raise ValueError(f"'{clave}' encuentra {len(encontradas)} categorias: {encontradas}")
    return encontradas[0]


INCLUIDOS = {buscar("restaurants"): "restaurante",
             buscar("cibercafe"): "bar"}

OPCIONALES = {buscar("take away"): "comida_rapida",
              buscar("discoteques"): "ocio_nocturno",
              buscar("geladeries"): "degustacion"}

d = censo[censo["Nom_Activitat"].isin(INCLUIDOS)].copy()
d["tipo_local"] = d["Nom_Activitat"].map(INCLUIDOS)

print("=== SELECCION ===")
for etiqueta, n in d["tipo_local"].value_counts().items():
    print(f"  {etiqueta:<14} {n:>5,}")
print(f"  {'TOTAL':<14} {len(d):>5,}")
print()
print("Fuera por decision, se reincorporan anadiendolas a INCLUIDOS:")
for actividad, etiqueta in OPCIONALES.items():
    print(f"  {etiqueta:<14} {int((censo['Nom_Activitat'] == actividad).sum()):>5,}")
print()
print(f"Fuera por no ser restauracion o ser alojamiento: "
      f"{len(censo) - len(d) - sum((censo['Nom_Activitat'] == a).sum() for a in OPCIONALES):,}")

=== SELECCION ===
  restaurante    4,430
  bar            4,273
  TOTAL          8,703

Fuera por decision, se reincorporan anadiendolas a INCLUIDOS:
  comida_rapida    778
  ocio_nocturno    387
  degustacion      148

Fuera por no ser restauracion o ser alojamiento: 848


## 3. Como esta el dato seleccionado

Antes de construir nada: que falta, que se repite y que no cuadra.

In [5]:
# 5. Nulos, y el marcador que no es nulo
#
# OJO: el censo NO deja celdas vacias en el nombre. Usa el literal `SN` --sense nom--, asi que un
# `isna()` dice que estan todos informados y es mentira.
campos = ["Nom_Local", "Nom_Activitat", "Nom_Barri", "Nom_Districte", "Nom_Via",
          "Num_Policia_Inicial", "Latitud", "Longitud", "Data_Revisio"]
tabla = pd.DataFrame({
    "nulos": [int(d[c].isna().sum()) for c in campos],
    "informados_%": [round(d[c].notna().mean() * 100, 1) for c in campos],
}, index=campos)
display(tabla)

sin_nombre = d["Nom_Local"].astype(str).str.strip().str.upper().eq("SN")
print(f"nombre = 'SN' (sin nombre): {int(sin_nombre.sum())} ({sin_nombre.mean():.1%})")
print()
print("Por tipo:")
print(d.assign(sn=sin_nombre).groupby("tipo_local")["sn"].agg(["size", "sum"]).to_string())

,nulos,informados_%
Nom_Local,0,100.0
Nom_Activitat,0,100.0
Nom_Barri,0,100.0
Nom_Districte,0,100.0
Nom_Via,0,100.0
Num_Policia_Inicial,0,100.0
Latitud,0,100.0
Longitud,0,100.0
Data_Revisio,0,100.0


nombre = 'SN' (sin nombre): 55 (0.6%)

Por tipo:
             size  sum
tipo_local            
bar          4273   36
restaurante  4430   19


In [6]:
# 6. Duplicados y cobertura territorial
print("=== IDENTIDAD ===")
print(f"filas                : {len(d):,}")
print(f"ID_Global distintos  : {d['ID_Global'].nunique():,}")
print(f"ID_Bcn_2016 repetidos: {int(d['ID_Bcn_2016'].duplicated().sum()):,}")
print()
# Un mismo portal puede tener varios locales: no es un duplicado, son negocios distintos.
misma_direccion = d.duplicated(subset=["Nom_Via", "Num_Policia_Inicial"], keep=False)
print(f"locales que comparten portal con otro: {int(misma_direccion.sum()):,}")
print(f"mismo portal Y mismo nombre           : "
      f"{int(d.duplicated(subset=['Nom_Via', 'Num_Policia_Inicial', 'Nom_Local']).sum()):,}"
      "   <- estos si son sospechosos")
print()
print("=== TERRITORIO ===")
print(f"barrios   : {d['Nom_Barri'].nunique()} de 73")
print(f"distritos : {d['Nom_Districte'].nunique()} de 10")
fuera = ((d["Latitud"] < 41.30) | (d["Latitud"] > 41.48) |
         (d["Longitud"] < 2.05) | (d["Longitud"] > 2.24))
print(f"coordenadas fuera de Barcelona: {int(fuera.sum())}")

=== IDENTIDAD ===
filas                : 8,703
ID_Global distintos  : 8,703
ID_Bcn_2016 repetidos: 597

locales que comparten portal con otro: 1,363
mismo portal Y mismo nombre           : 17   <- estos si son sospechosos

=== TERRITORIO ===
barrios   : 73 de 73
distritos : 10 de 10
coordenadas fuera de Barcelona: 0


In [7]:
# 7. Reparto por distrito y por barrio
por_distrito = (d.groupby("Nom_Districte")
                .agg(locales=("ID_Global", "size"),
                     bares=("tipo_local", lambda s: int((s == "bar").sum())),
                     restaurantes=("tipo_local", lambda s: int((s == "restaurante").sum())))
                .sort_values("locales", ascending=False))
por_distrito["bares_por_restaurante"] = (por_distrito.bares / por_distrito.restaurantes).round(2)
print("=== POR DISTRITO ===")
display(por_distrito)

print("=== LOS DIEZ BARRIOS CON MAS LOCALES ===")
display(d.groupby(["Nom_Districte", "Nom_Barri"])
        .agg(locales=("ID_Global", "size"),
             bares=("tipo_local", lambda s: int((s == "bar").sum())),
             restaurantes=("tipo_local", lambda s: int((s == "restaurante").sum())))
        .sort_values("locales", ascending=False).head(10))

=== POR DISTRITO ===


,locales,bares,restaurantes,bares_por_restaurante
Nom_Districte,,,,
Eixample,2422,990,1432,0.69
Sant Martí,1188,690,498,1.39
Ciutat Vella,1062,402,660,0.61
Sants-Montjuïc,835,421,414,1.02
Sarrià-Sant Gervasi,623,258,365,0.71
Sant Andreu,576,354,222,1.59
Nou Barris,566,393,173,2.27
Gràcia,565,276,289,0.96
Horta-Guinardó,440,304,136,2.24


=== LOS DIEZ BARRIOS CON MAS LOCALES ===


locales  bares  \
Nom_Districte       Nom_Barri                                         
Eixample            la Dreta de l'Eixample               657    244   
                    l'Antiga Esquerra de l'Eixample      605    192   
Gràcia              la Vila de Gràcia                    383    175   
Eixample            la Nova Esquerra de l'Eixample       348    158   
                    Sant Antoni                          332    150   
Ciutat Vella        el Raval                             328    158   
Eixample            la Sagrada Família                   306    160   
Sarrià-Sant Gervasi Sant Gervasi - Galvany               298    109   
Les Corts           les Corts                            292    107   
Ciutat Vella        el Barri Gòtic                       284     88   

                                                     restaurantes  
Nom_Districte       Nom_Barri                                      
Eixample            la Dreta de l'Eixample                    413  
                    l'Antiga Esquerra de l'Eixample           413  
Gràcia              la Vila de Gràcia                         208  
Eixample            la Nova Esquerra de l'Eixample            190  
                    Sant Antoni                               182  
Ciutat Vella        el Raval                                  170  
Eixample            la Sagrada Família                        146  
Sarrià-Sant Gervasi Sant Gervasi - Galvany                    189  
Les Corts           les Corts                                 185  
Ciutat Vella        el Barri Gòtic                            196

## 4. Que queda pendiente

- **Decidir si entran las tres categorias opcionales** (take away, ocio nocturno, degustacion).
- **Que hacer fuera de Barcelona ciudad.** El censo solo cubre la ciudad. Para el resto de la
  provincia la unica fuente es OSM, con los problemas que se han visto, asi que la capa sera
  hibrida y hay que explicarlo en la web.

Cuando eso este cerrado: script en `pipeline/gold/` y `gold` generado desde el.
